In [19]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

def search_places(query: str, size: int = 15, category_code: str = "") -> list[dict]:
    """카카오 로컬 키워드 검색으로 장소를 찾는다.

    카카오는 매장명·카테고리·태그를 매칭하므로 검색어는
    '지역 + 짧은 키워드' 형태로 만드는 게 좋다.
      좋은 예: '성수동 카페', '성수동 노트북 카페'
      나쁜 예: '성수 조용히 작업하기 좋은 카페' (서술형은 빈 결과가 나기 쉬움)

    Args:
        query: 검색어.
        size: 결과 개수 (1~15).
        category_code: 카테고리 필터. 'CE7'=카페, 'FD6'=음식점,
                        'AT4'=관광명소, 'CT1'=문화시설.
    Returns:
        장소 팩트 리스트 (이름·주소·전화·좌표·url).
    """
    params = {"query": query, "size": size}
    if category_code:
        params["category_group_code"] = category_code

    resp = requests.get(
        "https://dapi.kakao.com/v2/local/search/keyword.json",
        headers={"Authorization": f"KakaoAK {os.environ['KAKAO_REST_API_KEY']}"},
        params=params,
        timeout=5,
    )
    resp.raise_for_status()

    return [
        {
            "name": d["place_name"],
            "category": d["category_name"],
            "address": d["road_address_name"] or d["address_name"],
            "phone": d["phone"],
            "x": d["x"],
            "y": d["y"],
            "url": d["place_url"],
        }
        for d in resp.json()["documents"]
    ]

In [23]:
import os
from dotenv import load_dotenv
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.genai import types

load_dotenv()

agent = LlmAgent(
    name="basic_agent",
    model="gemini-3.1-flash-lite",
    tools = [search_places],
    instruction="너는 친절한 어시스턴트다.",
)

runner = InMemoryRunner(agent=agent, app_name="demo")

async def ask(text: str):
    session = await runner.session_service.create_session(
        app_name="demo", user_id="u1", session_id="s1"
    )
    msg = types.Content(role="user", parts=[types.Part(text=text)])

    async for event in runner.run_async(user_id="u1", session_id="s1", new_message=msg):
        if event.is_final_response() and event.content:
            print(event.content.parts[0].text)

await ask("너가 가지고 있는 툴은 뭐뭐 있어? 그거에 대한 정보는")

저는 현재 **`search_places`**라는 하나의 도구를 가지고 있습니다. 이 도구는 **카카오 로컬(Kakao Local) API**를 활용하여 장소 정보를 검색하는 기능을 제공합니다.

상세 정보는 다음과 같습니다:

### 도구 이름: `search_places`
*   **설명**: 사용자가 검색어를 입력하면, 카카오 로컬 데이터를 기반으로 매장명, 주소, 전화번호, 좌표, 상세 페이지 URL 등의 장소 정보를 찾아줍니다.
*   **주요 기능**:
    *   **지역 기반 검색**: '지역 + 키워드' 형식(예: '성수동 카페', '강남 맛집')으로 검색하면 가장 정확한 결과를 얻을 수 있습니다.
    *   **카테고리 필터**: 검색 시 특정 카테고리를 지정하여 결과를 제한할 수 있습니다.
        *   `CE7`: 카페
        *   `FD6`: 음식점
        *   `AT4`: 관광명소
        *   `CT1`: 문화시설
    *   **결과 개수 조절**: 필요에 따라 한 번에 가져올 검색 결과의 개수(1~15개)를 지정할 수 있습니다.

---

**사용 팁:**
*   너무 긴 서술형 문장보다는 **"지역명 + 짧은 키워드"**로 검색하시는 것이 훨씬 결과가 좋습니다.
    *   *좋은 예:* "성수동 노트북 카페", "홍대 맛집"
    *   *피해야 할 예:* "성수동에서 조용하게 노트북을 하기 좋은 카페를 찾아줘" (서술형은 검색 정확도가 낮아질 수 있습니다.)

이 도구를 사용해 도움이 필요하신 장소 검색이 있다면 언제든 말씀해 주세요!
